### RAG (PDF) — LangChain 0.3+ + Ollama (Llama 3.2)This notebook implements a very simple PDF-only RAG pipeline.

In [1]:
# Imports
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

In [2]:
load_dotenv()

False

In [3]:
# CONFIG
PDF_PATH = "./docs/myfile.pdf"
CHROMA_DIR = "./my_chroma_db"

EMBED_MODEL = "granite-embedding:latest"
LLM_MODEL = "gpt-oss:120b-cloud"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200  

In [4]:
# LOAD PDF
loader = PyPDFLoader(PDF_PATH)

docs = loader.load()
print('Documents loaded:', len(docs))
print(docs[2].page_content)
print(docs[2].metadata)


Documents loaded: 131
Section on :
Course Introduction
{'producer': 'Microsoft® PowerPoint® 2021', 'creator': 'Microsoft® PowerPoint® 2021', 'creationdate': '2025-05-05T22:00:54+05:30', 'title': '', 'author': 'Anisha Trisal', 'keywords': '', 'moddate': '2025-05-05T22:00:54+05:30', 'source': './docs/myfile.pdf', 'total_pages': 131, 'page': 2, 'page_label': '3'}


In [5]:
# CHUNK
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_documents(docs)

print('Document chunks created:', len(chunks))

print('Document chunks :', chunks[0])


Document chunks created: 137
Document chunks : page_content='Building AWS AI Agents 
using 
Bedrock Multi-Agent Framework' metadata={'producer': 'Microsoft® PowerPoint® 2021', 'creator': 'Microsoft® PowerPoint® 2021', 'creationdate': '2025-05-05T22:00:54+05:30', 'title': '', 'author': 'Anisha Trisal', 'keywords': '', 'moddate': '2025-05-05T22:00:54+05:30', 'source': './docs/myfile.pdf', 'total_pages': 131, 'page': 0, 'page_label': '1'}


In [6]:
# VECTOR DB with OllamaEmbeddings embeddings
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

vector_db = Chroma(
    collection_name="pdf_collection",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)

vector_db.add_documents(chunks)

['45df62d8-d3d5-4eb5-bd3a-182b8d478d70',
 'c96f6f42-b68c-46d9-8353-61afa096c8f8',
 '5499257c-415a-40bb-be41-bc7c9b0176ad',
 '577a2589-8db4-4f20-ad7e-d6d84f5809d6',
 '6ffe89c6-283b-4680-ade6-50fbffedece1',
 'dbb27d19-e8bd-4a8c-94be-e84e84e7e3bd',
 'be79329b-fbf2-4997-8b06-53538e84eddf',
 'be107a24-c55d-4e1d-9b35-d0dc88448923',
 'e70d5f4e-5e8d-4642-82b9-4590f615932c',
 'f2707c38-b25f-4f2b-b21d-bfa928078d1b',
 '27ebde75-9271-4c5e-b65d-bbdbb9354136',
 'daca54c7-f886-456d-92dd-2ac9375b989f',
 '671d1444-81f2-40e1-8a8a-87c9ddbd7b36',
 '8b9bc9f4-3eb7-4e04-9e7a-fbda0f9c177f',
 'a5bf8b45-e89e-4211-b1f7-cb7a9f6d67fb',
 'e237c0f5-1935-4846-8623-628c4f686dbc',
 'ce39da03-25ae-4045-a5c5-5b6d39c56690',
 '4e5fd994-4efa-4112-a12b-f9ace0bcce43',
 'd87f01fb-0bcd-41bd-9b1d-a5f7de1abb14',
 'c9a727eb-b59f-4eac-9d53-a4f50d3c33ce',
 '94065cb0-1c58-4bd9-bffc-244dc041f21c',
 '70c2cd90-9b27-4e60-834f-7af2da1e57b9',
 '89effe44-cc68-4186-b26d-ad230015c4d8',
 '55d0d174-cd58-455f-8dee-122eb9b595bb',
 'b4f87762-539a-

In [7]:
# LLM (Llama 3.2)
llm = ChatOllama(model=LLM_MODEL)

In [ ]:
# PROMPT (new RAG pattern)
prompt = ChatPromptTemplate.from_template(
    """
Use the context below to answer the question.

If you don't know the answer, say you don't know. Don't try to make up an answer.

Context:
{context}

Question:
{question}
""")

In [12]:
# FINAL RAG FUNCTION (manual RAG)
def rag(question):
    # 1. retrieve top chunks
    
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})  #R
    docs = retriever.invoke(question)
    

    # 2. combine retrieved chunks
    context = "\n\n".join([d.page_content for d in docs])

    # 3. create prompt
    final_prompt = prompt.invoke({"context": context, "question": question})

    # 4. send to Llama 3.2
    return llm.invoke(final_prompt)


In [13]:
# CHAT LOOP
while True:
    q = input("You: ")
    if q.lower() == "exit":
        break

    answer = rag(q)
    print("\nAnswer:\n", answer.content)


Answer:
 The Prime Minister of India is **Narendra Modi**.
